In [ ]:
# PCA class to run IncrementalPCA on the df_merge output (or on a parquet produced by 1_dataset_processing.ipynb)
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from sklearn.decomposition import IncrementalPCA
import gc
from typing import Optional


class PCA:
    """Incremental PCA helper that can fit from a parquet file or an in-memory DataFrame

    Usage:
      - If you already have a parquet produced by the dataset processing step: use
            p = PCA(n_components=100)
            p.fit_transform_parquet(parquet_path, output_path)

      - If you have `df_merge` loaded in memory (the merged residue + esm embeddings):
            p = PCA(n_components=100)
            p.fit_from_dataframe(df_merge)
            reduced_df = p.transform_dataframe(df_merge)

    Notes:
      - The class looks for numeric embedding columns starting with `numeric_prefix` (default `esm_`).
      - Uses IncrementalPCA so it can handle large datasets in chunks.
    """

    def __init__(self, n_components: int = 685, chunk_size: int = 50_000, numeric_prefix: str = "esm_"):
        self.n_components = n_components
        self.chunk_size = chunk_size
        self.numeric_prefix = numeric_prefix
        self.ipca: Optional[IncrementalPCA] = None
        self.numeric_cols = None
        self.id_cols = None

    def _init_ipca(self):
        if self.ipca is None:
            self.ipca = IncrementalPCA(n_components=self.n_components)

    def _infer_cols_from_parquet(self, parquet_path: str):
        parquet_file = pq.ParquetFile(parquet_path)
        self.numeric_cols = [x for x in parquet_file.schema.names if x.startswith(self.numeric_prefix)]
        self.id_cols = [x for x in parquet_file.schema.names if not x.startswith(self.numeric_prefix)]
        return parquet_file

    def _infer_cols_from_dataframe(self, df: pd.DataFrame):
        self.numeric_cols = [c for c in df.columns if c.startswith(self.numeric_prefix)]
        self.id_cols = [c for c in df.columns if not c.startswith(self.numeric_prefix)]

    def fit_from_parquet(self, parquet_path: str):
        """Fit IncrementalPCA by streaming parquet batches."""
        parquet_file = self._infer_cols_from_parquet(parquet_path)
        self._init_ipca()

        print("Fitting PCA from parquet...")
        batch_count = 0
        for batch in parquet_file.iter_batches(batch_size=self.chunk_size):
            chunk = batch.to_pandas()[self.numeric_cols].values.astype(np.float16)
            self.ipca.partial_fit(chunk)
            batch_count += 1
            if batch_count % 10 == 0:
                print(f"  Fitted {batch_count * self.chunk_size:,} rows...")
            del chunk
            gc.collect()

        print(f"PCA fit. Variance explained: {self.ipca.explained_variance_ratio_.sum():.2f}")

    def transform_parquet_to_parquet(self, parquet_path: str, output_path: str, compression: str = "snappy"):
        """Transform the parquet using the already-fitted PCA and write reduced parquet."""
        if self.ipca is None:
            raise ValueError("PCA not fitted yet. Call fit_from_parquet or fit_from_dataframe first.")

        parquet_file = pq.ParquetFile(parquet_path)
        # ensure columns known
        if self.numeric_cols is None or self.id_cols is None:
            self._infer_cols_from_parquet(parquet_path)

        writer = None
        batch_count = 0
        for batch in parquet_file.iter_batches(batch_size=self.chunk_size):
            df = batch.to_pandas()
            ids_df = df[self.id_cols].reset_index(drop=True)

            reduced_features = self.ipca.transform(df[self.numeric_cols].values.astype(np.float16))
            reduced_features = reduced_features.astype(np.float16)

            reduced_df = pd.DataFrame(
                reduced_features,
                columns=[f'PC{i+1}' for i in range(self.n_components)]
            )

            final_chunk = pd.concat([ids_df, reduced_df], axis=1)
            table = pa.Table.from_pandas(final_chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_path, table.schema, compression=compression)
            writer.write_table(table)

            batch_count += 1
            if batch_count % 10 == 0:
                print(f"  Transformed {batch_count * self.chunk_size:,} rows...")

            del df, ids_df, reduced_features, reduced_df, final_chunk, table
            gc.collect()

        if writer is not None:
            writer.close()
        print("Transformation complete and saved to:", output_path)

    def fit_transform_parquet(self, parquet_path: str, output_path: str, compression: str = "snappy"):
        """Convenience method: fit then transform on parquet inputs."""
        self.fit_from_parquet(parquet_path)
        self.transform_parquet_to_parquet(parquet_path, output_path, compression=compression)

    def fit_from_dataframe(self, df: pd.DataFrame):
        """Fit using an in-memory DataFrame (works in incremental chunks)."""
        self._infer_cols_from_dataframe(df)
        self._init_ipca()

        print("Fitting PCA from DataFrame...")
        n = len(df)
        for start in range(0, n, self.chunk_size):
            chunk = df[self.numeric_cols].iloc[start:start + self.chunk_size].values.astype(np.float16)
            self.ipca.partial_fit(chunk)
            if ((start // self.chunk_size) + 1) % 10 == 0:
                print(f"  Fitted {start + len(chunk):,} rows...")
            del chunk
            gc.collect()

        print(f"PCA fit. Variance explained: {self.ipca.explained_variance_ratio_.sum():.4f}")

    def transform_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform an in-memory DataFrame (returns reduced DataFrame with PC columns)."""
        if self.ipca is None:
            raise ValueError("PCA not fitted yet. Call fit_from_dataframe or fit_from_parquet first.")
        if self.numeric_cols is None:
            self._infer_cols_from_dataframe(df)

        reduced_features = self.ipca.transform(df[self.numeric_cols].values.astype(np.float16))
        reduced_features = reduced_features.astype(np.float16)
        reduced_df = pd.DataFrame(reduced_features, columns=[f'PC{i+1}' for i in range(self.n_components)])
        final = pd.concat([df[self.id_cols].reset_index(drop=True), reduced_df], axis=1)
        return final


print("PCA helper class loaded. Use PCA(...) to create an instance.")

In [2]:
import pandas as pd

df = pd.read_parquet('/kaggle/input/datasets/yangxinzhandu/model-dataset-pca-parquet/model_dataset_pca.parquet')

In [6]:
df.drop(columns=['receptor_chain', 'ligand_id', 'position'])

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,pdb_id,residue,isbinding,PC1,PC2,PC3,PC4,PC5,PC6,PC7,...,PC676,PC677,PC678,PC679,PC680,PC681,PC682,PC683,PC684,PC685
0,13pk,E,0,0.265137,-0.257324,-0.032990,0.048096,-0.182861,-0.096619,-0.050568,...,-0.001503,-0.011368,0.014687,0.010735,-0.028381,-0.008575,-0.006489,-0.035553,0.038300,-0.011383
1,13pk,K,0,-0.041809,-0.103271,-0.422119,-0.231934,-0.435303,-0.352295,0.002403,...,-0.005199,-0.013786,-0.007591,-0.019867,0.010002,0.001771,-0.027481,0.001056,-0.022552,-0.017471
2,13pk,K,0,0.273682,0.022079,-0.378174,-0.311523,-0.307617,-0.103088,0.040955,...,-0.001213,-0.013924,-0.006695,-0.020660,0.016907,-0.020111,-0.025574,-0.021957,0.006382,0.029221
3,13pk,S,0,-0.022125,-0.007603,-0.096802,-0.050110,-0.356689,-0.127686,0.041748,...,-0.006538,0.017517,-0.025162,-0.007828,-0.003050,0.029434,-0.018036,-0.016312,-0.022614,0.012321
4,13pk,I,0,-0.431396,0.582520,0.064819,-0.120850,0.117920,0.022064,-0.127686,...,0.017593,0.003042,-0.001300,-0.019302,0.033661,-0.029266,0.042084,0.024582,0.005203,0.028687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5057166,9v59,T,0,0.591309,-0.127686,0.091980,-0.084839,0.005505,0.009895,-0.181274,...,-0.006927,-0.001328,0.008110,0.019150,0.019180,-0.010544,0.021805,0.012901,0.014450,0.040710
5057167,9v59,L,0,0.153320,0.418701,-0.051880,-0.179688,-0.121338,0.070618,-0.169556,...,-0.017578,0.002501,-0.028824,0.026810,-0.009178,-0.014580,0.000294,0.002579,0.010841,0.002974
5057168,9v59,E,0,0.726562,-0.141113,0.085632,-0.081238,-0.033356,0.048065,-0.100769,...,0.000205,-0.016510,0.007080,-0.003624,0.002323,-0.000090,0.002977,-0.019455,-0.001738,-0.009079
5057169,9v59,I,0,0.636230,0.303467,0.010826,-0.075745,-0.179321,0.099854,-0.172485,...,-0.005085,-0.004753,0.011223,0.002884,-0.003626,0.021210,0.017395,-0.007828,0.005287,0.016266
